In [ ]:
from utils import *

In [ ]:
CSV = Path("/path/to/observation_list.csv")
df = pd.read_csv(CSV)

jobs = []
for _, row in df.iterrows():
    date = pd.to_datetime(row["Enh date"]).strftime("%Y-%m-%d")
    time = str(row["Enh time"]).strip()

    if len(time) > 9:
        start_time = time[:8]
        end_time = time[11:]
    else:
        start_time = time[:8]
        end_time = time[:8]

    jobs.append({
        "obs_id": row["IRIS flares"],
        "start": f"{date}T{start_time}",
        "end": f"{date}T{end_time}"
    })



In [ ]:
# Use your paths
OUT_FILE = Path("/path/to/output_file.tsv") 
OBS_FOLDER = Path("/path/to/observation_folders")


# Select which alignment method to use
method = 'chi2'
#method = 'phase'

# Initialize some variables
obs = iris_sji = iris_map = None
aia_maps = None
aia_seq = []
iris_seq = []
matched_maps = []
results = []

# Loop through the data
for job_idx, job in enumerate(tqdm(jobs)):

    obs_id = job["obs_id"]
    start_time = job["start"]
    end_time = job["end"]

    print(f"============ Observation ID: {obs_id} ============\n")
    print(f"Window: {start_time} to {end_time}")

    obs_folder = OBS_FOLDER / obs_id
    obs_folder.mkdir(parents=True, exist_ok=True)


    # Load IRIS data
    obs = ir.observation(str(obs_folder))
    iris_sji = obs.sji[0]


    # Load AIA data
    aia_observations = list(obs_folder.glob("*.image.fits"))
    aia_observations.sort()
    aia_maps = Map(aia_observations, sequence=True, allow_errors=True)


    # Get timestamps and search for matching frames between IRIS and AIA
    iris_times_s = np.array(iris_sji.get_timestamps()) 
    aia_times_s = np.array([m.date.unix for m in aia_maps])
    start_s = Time(start_time).unix
    end_s = Time(end_time).unix

    matching_frames = find_matching_frames(iris_times_s, aia_times_s, start_s, end_s)

    if not matching_frames:
        print(f"[{job_idx}] No matching frames...\n")
        write_error_row(OUT_FILE, obs_id,
                        "No matching AIA and IRIS frames found.")
        continue

    print(f"{len(matching_frames)} Matching indices: \n")
    print(matching_frames)


    # Save selected frames in the lists to use for the alignment
    iris_seq = []
    aia_seq = []
    
    for aia_frame, iris_frame in matching_frames:
        aia_seq.append(aia_maps[aia_frame])

        iris_map = iris_to_sunpy_map(iris_sji, iris_frame)
        iris_seq.append(iris_map)

    matched_maps = [(aia_map, iris_map) for aia_map, iris_map in zip(aia_seq, iris_seq)]


    # Alignment of the frames, stores results in a list
    results = Parallel(n_jobs=-1, backend="threading")(
        delayed(align_aia_iris)(aia_map.data, aia_map.meta, iris_map.data, iris_map.meta, method=method)
        for (aia_map, iris_map) in matched_maps
    )


    # Writes the results to the output file
    write_to_file(
        out_file=OUT_FILE,
        observation_id=obs_id,
        matches=matching_frames,
        results=results,
        iris_times_s=iris_times_s,
        aia_times_s=aia_times_s
    )
    print(f"Saved results to {OUT_FILE}\n")



    # clear memory
    obs = None
    iris_sji = None
    iris_map = None
    aia_observations = []
    aia_maps = None
    aia_seq = []
    iris_seq = []

    matched_maps = []
    results = []

    iris_times_s = None
    aia_times_s = None
    matching_frames = None

    gc.collect() 

print("=============FINISHED===============")